# CRT matmul

FP64-accurate matmul using only int8 arithmetic. From [Ozaki scheme II](https://arxiv.org/abs/2504.08009).

| Step | Operation | Types |
|------|-----------|-------|
| 1 | Scale rows of $A$, cols of $B$ | float64 -> int64 |
| 2 | Reduce mod primes $p_i < 128$ | int64 -> int8 |
| 3 | Matmul per prime | int8 x int8 -> int32 |
| 4 | CRT reconstruction | int32 -> int64 -> float64 |
| 5 | Rescale | float64 |

In [ ]:
import os
import sys

if "COLAB_GPU" in os.environ:
    os.system("git clone https://github.com/PritRaj1/tensor_inv.git 2>/dev/null")
    sys.path.insert(0, "/content/tensor_inv/src")
else:
    src = "src" if os.path.isdir("src") else "../src"
    sys.path.insert(0, src)

In [ ]:
import torch
import math
import time
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120

## Chinese Remainder Theorem

Given coprime $m_1, \ldots, m_L$ and residues $r_i = x \bmod m_i$, CRT recovers $x$ uniquely if $|x| < \prod m_i / 2$.

In [ ]:
primes = [3, 5, 7, 11, 13]
M = math.prod(primes)
x = 12345
residues = [x % p for p in primes]

# reconstruct
x_rec = 0
for r, m in zip(residues, primes):
    Mi = M // m
    x_rec += r * Mi * pow(Mi, -1, m)
x_rec %= M

print(f"x = {x}, residues = {residues}")
print(f"recovered = {x_rec}, match: {x == x_rec}")
print(f"product of primes = {M}, covers |x| < {M // 2}")

## Scaling to integers

Normalize rows of $A$ and columns of $B$ by their max absolute value, then scale to the int52 range ($2^{52}$, the FP64 mantissa width). Round to nearest integer.

In [ ]:
SCALE = float(2**52)

torch.manual_seed(42)
A = torch.randn(4, 3, dtype=torch.float64)
B = torch.randn(3, 5, dtype=torch.float64)

row_max = A.abs().amax(dim=1).clamp(min=1e-300)
col_max = B.abs().amax(dim=0).clamp(min=1e-300)

A_int = torch.round(A / row_max.unsqueeze(1) * SCALE).to(torch.int64)
B_int = torch.round(B / col_max.unsqueeze(0) * SCALE).to(torch.int64)

print(f"A range: [{A.min():.2f}, {A.max():.2f}]")
print(f"A_int range: [{A_int.min()}, {A_int.max()}]")
print(f"fits in int52: {A_int.abs().max() <= 2**52}")

## Modular reduction

Reduce each int64 entry mod small primes ($< 128$, fits int8). Each dot product in the matmul is now a dot product of int8 values mod $p$.

In [ ]:
# fmt: off
primes = [3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53,
          59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127]
# fmt: on

# how many primes needed?
k = A.shape[1]
target = 2 * k * SCALE * SCALE
prod = 1
for i, p in enumerate(primes):
    prod *= p
    if prod > target:
        n_mod = i + 1
        break

moduli = primes[:n_mod]
print(f"inner dim k={k}, need {n_mod} primes: {moduli}")
print(f"product covers: {math.prod(moduli):.2e} > {target:.2e}")

## Per-prime matmul + CRT

Matmul in each residue channel independently ($\text{int8} \times \text{int8} \to \text{int32}$), then reconstruct the exact int64 result via CRT.

In [ ]:
# per-prime matmul
residues = []
for m in moduli:
    a_mod = (A_int % m).to(torch.int8)
    b_mod = (B_int % m).to(torch.int8)
    c = a_mod.int() @ b_mod.int()
    residues.append(c % m)


# CRT reconstruction
def crt_scalar(rs, mods):
    M = math.prod(mods)
    x = 0
    for r, m in zip(rs, mods):
        Mi = M // m
        x += r * Mi * pow(Mi, -1, m)
    x %= M
    return x - M if x > M // 2 else x


scale_sq = SCALE**2
rows, cols = residues[0].shape
C_crt = torch.empty(rows, cols, dtype=torch.float64)
for i in range(rows):
    for j in range(cols):
        rs = [residues[p][i, j].item() for p in range(n_mod)]
        C_crt[i, j] = crt_scalar(rs, moduli) / scale_sq * row_max[i] * col_max[j]

C_ref = A @ B
print(f"max abs error: {(C_crt - C_ref).abs().max():.2e}")

## Accuracy

In [ ]:
from tensor_inv import crt_matmul

sizes = [32, 64, 128, 256]

print(f"{'n':>6} {'max_abs_err':>14} {'max_rel_err':>14}")
print("-" * 38)

for n in sizes:
    torch.manual_seed(42)
    A = torch.randn(n, n, dtype=torch.float64)
    B = torch.randn(n, n, dtype=torch.float64)

    C = crt_matmul(A, B)
    ref = A @ B

    abs_err = (C - ref).abs().max().item()
    nz = ref.abs() > 1e-12
    rel_err = ((C - ref).abs()[nz] / ref.abs()[nz]).max().item()

    print(f"{n:>6} {abs_err:>14.3e} {rel_err:>14.3e}")

## Number of primes vs inner dimension

More primes needed as $k$ grows, since the dot product range is $\sim 2k \cdot (2^{52})^2$.

In [ ]:
from tensor_inv.crt_matmul import _n_moduli, _PRIMES

ks = [16, 32, 64, 128, 256, 512, 1024]
print(f"{'k':>6} {'primes':>8} {'product (bits)':>16}")
print("-" * 34)
for k in ks:
    n = _n_moduli(k)
    bits = math.log2(math.prod(_PRIMES[:n]))
    print(f"{k:>6} {n:>8} {bits:>16.1f}")

## Benchmark

CRT matmul vs `torch.matmul` (FP64).

In [ ]:
sizes = [32, 64, 128, 256]

print(f"{'n':>6} {'torch (ms)':>12} {'CRT (ms)':>12} {'ratio':>8}")
print("-" * 42)

for n in sizes:
    torch.manual_seed(0)
    A = torch.randn(n, n, dtype=torch.float64)
    B = torch.randn(n, n, dtype=torch.float64)

    # warmup
    for _ in range(3):
        _ = A @ B
        _ = crt_matmul(A, B)

    t0 = time.perf_counter()
    for _ in range(5):
        _ = A @ B
    t_torch = (time.perf_counter() - t0) / 5 * 1000

    t0 = time.perf_counter()
    for _ in range(5):
        _ = crt_matmul(A, B)
    t_crt = (time.perf_counter() - t0) / 5 * 1000

    print(f"{n:>6} {t_torch:>12.2f} {t_crt:>12.2f} {t_crt / t_torch:>7.1f}x")